# ☕ Capstone Data Analysis: Coffee Shop Sales
## Tahap 4: Feature Engineering

---

**Tujuan Notebook Ini:**
- Membuat fitur-fitur baru yang dapat meningkatkan kualitas analisis bisnis
- Mempermudah proses visualisasi maupun dashboard
- Menambahkan dimensi analisis yang sebelumnya tidak tersedia

**Dataset Input:** `processed/coffee_shop_sales_clean.csv`
**Dataset Output:** `processed/coffee_shop_sales_featured.csv`

---

# 1. Objective

Feature Engineering adalah proses membuat fitur-fitur baru dari data yang sudah ada untuk meningkatkan kualitas analisis bisnis. Dalam konteks Coffee Shop Sales, feature engineering bertujuan untuk:

1. **Menambahkan dimensi temporal** - ekstrak bulan, kuartal, hari, jam dari timestamp untuk analisis tren
2. **Membuat segmentasi bisnis** - kategorisasi pelanggan, produk, dan transaksi berdasarkan nilai
3. **Membuat flag indikator** - penanda weekend, peak hour, dan kondisi bisnis lainnya
4. **Mempermudah visualisasi** - fitur yang sudah terkategorisasi lebih mudah diplot
5. **Menyiapkan data untuk modeling** - fitur yangsiap digunakan untuk machine learning

Setiap fitur baru yang dibuat harus memiliki **manfaat analisis yang jelas** dan tidak boleh membuat fitur yang tidak berguna.

---
# 2. Business Questions

### Q1: Fitur baru apa saja yang perlu dibuat?
Berdasarkan kebutuhan analisis bisnis coffee shop:
- **Temporal Features**: Month, Quarter, Day of Week, Hour - untuk analisis tren waktu
- **Behavioral Features**: Weekend Flag, Customer Segment - untuk analisis perilaku
- **Product Features**: Price Category, Basket Size - untuk analisis produk
- **Financial Features**: Discount Impact - untuk analisis profitabilitas

### Q2: Mengapa fitur tersebut penting?
- **Temporal**: Mengetahui kapan penjualan paling tinggi/rendah
- **Behavioral**: Memahami pola belanja pelanggan
- **Product**: Mengkategorikan produk berdasarkan harga dan volume
- **Financial**: Mengukur dampak diskon terhadap pendapatan

### Q3: Bagaimana fitur baru membantu analisis bisnis?
- Memungkinkan analisis **tren musiman** (bulanan, kuartalan)
- Memungkinkan **segmentasi pelanggan** berdasarkan perilaku
- Memungkinkan **analisis performa produk** berdasarkan kategori harga
- Memungkinkan **analisis dampak promosi** terhadap penjualan

### Q4: Apakah fitur baru dapat meningkatkan insight yang diperoleh?
**Ya.** Fitur baru akan memberikan insight seperti:
- Jam berapa peak hour terjadi
- Hari apa penjualan paling tinggi
- Segmen mana yang paling menguntungkan
- Bagaimana dampak diskon terhadap revenue

---
# 3. Load Clean Dataset

Memuat dataset hasil Data Cleaning dari tahap sebelumnya.

In [ ]:
# ============================================================
# Import Library
# ============================================================
import pandas as pd
import numpy as np
import os
import warnings

# Konfigurasi
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Library berhasil diimport.")

In [ ]:
# ============================================================
# Load Clean Dataset
# ============================================================
FILE_PATH = '../processed/coffee_shop_sales_clean.csv'

df = pd.read_csv(FILE_PATH)

# Konversi timestamp ke datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

print("=" * 70)
print(" CLEAN DATASET LOADED")
print("=" * 70)
print(f"\nFile: {FILE_PATH}")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\n5 Baris Pertama:")
df.head()

In [ ]:
# ============================================================
# Info Dataset
# ============================================================
print("=" * 70)
print(" INFORMASI DATASET")
print("=" * 70)
df.info()

print(f"\nMissing Values: {df.isnull().sum().sum():,}")
print(f"Duplicate Rows: {df.duplicated().sum():,}")

---
# 4. Feature Engineering Plan

Berikut adalah tabel rencana pembuatan fitur baru:

| New Feature | Source Column | Purpose | Business Value |
|-------------|---------------|---------|----------------|
| `month` | `timestamp` | Analisis tren bulanan | Mengidentifikasi musim penjualan (high/low season) |
| `quarter` | `timestamp` | Analisis kuartalan | Pelaporan bisnis quarterly dan perbandingan performa |
| `day_of_week` | `timestamp` | Analisis pola mingguan | Mengetahui hari tersibuk dan strategi staffing |
| `hour` | `timestamp` | Analisis peak hours | Mengoptimalkan jam operasional dan promosi |
| `is_weekend` | `day_of_week` | Perbandingan weekday vs weekend | Strategi promosi berbeda untuk weekday/weekend |
| `price_category` | `unit_price` | Segmentasi produk | Analisis performa per kategori harga (Low/Medium/Premium) |
| `basket_size` | `quantity` | Analisis volume belanja | Memahami pola pembelian (satuan vs bulk) |
| `customer_segment` | `total_amount` per customer | Segmentasi pelanggan | Strategi loyalitas dan personalisasi |
| `discount_impact` | `discount_applied`, `total_amount` | Analisis dampak diskon | Mengukur efektivitas promosi |
| `revenue_category` | `total_amount` | Kategorisasi transaksi | Analisis distribusi pendapatan per kategori |

### Prinsip Feature Engineering:
1. **Setiap fitur harus memiliki manfaat analisis yang jelas**
2. **Jangan membuat fitur yang redundan** dengan kolom yang sudah ada
3. **Gunakan domain knowledge** untuk menentukan threshold
4. **Dokumentasikan setiap keputusan** pembuatan fitur

---
# 5. Create New Features

Membuat fitur-fitur baru secara bertahap dengan penjelasan bisnis.

In [ ]:
# ============================================================
# Buat Salinan Dataset untuk Feature Engineering
# ============================================================
df_feat = df.copy()

print("=" * 70)
print(" FEATURE ENGINEERING")
print("=" * 70)
print(f"\nDataset asli: {df.shape[0]:,} baris × {df.shape[1]} kolom")
print(f"Memori: {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

---
## Feature 1: Month

Mengekstrak bulan dari kolom `timestamp`.

**Manfaat Bisnis:**
- Analisis tren penjualan bulanan
- Identifikasi musim penjualan (high season vs low season)
- Perencanaan stok dan promosi bulanan

In [ ]:
# ============================================================
# Feature 1: Month
# ============================================================

print("[1] Membuat fitur: month")

# Ekstrak bulan dari timestamp
df_feat['month'] = df_feat['timestamp'].dt.month

# Buat label bulan untuk visualisasi
month_names = {1: 'January', 2: 'February', 3: 'March', 4: 'April',
               5: 'May', 6: 'June', 7: 'July', 8: 'August',
               9: 'September', 10: 'October', 11: 'November', 12: 'December'}
df_feat['month_name'] = df_feat['month'].map(month_names)

print(f"    Tipe data: {df_feat['month'].dtype}")
print(f"    Unique values: {sorted(df_feat['month'].unique())}")
print(f"    Distribusi:")
for month, count in df_feat['month_name'].value_counts().sort_index().items():
    print(f"      {month}: {count:,} transaksi")

---
## Feature 2: Quarter

Mengelompokkan transaksi ke dalam kuartal (Q1-Q4).

**Manfaat Bisnis:**
- Pelaporan bisnis quarterly (Q1: Jan-Mar, Q2: Apr-Jun, Q3: Jul-Sep, Q4: Oct-Dec)
- Perbandingan performa antar kuartal
- Perencanaan anggaran tahunan

In [ ]:
# ============================================================
# Feature 2: Quarter
# ============================================================

print("[2] Membuat fitur: quarter")

# Hitung kuartal menggunakan pandas
df_feat['quarter'] = df_feat['timestamp'].dt.quarter

# Buat label kuartal
quarter_labels = {1: 'Q1 (Jan-Mar)', 2: 'Q2 (Apr-Jun)', 3: 'Q3 (Jul-Sep)', 4: 'Q4 (Oct-Dec)'}
df_feat['quarter_label'] = df_feat['quarter'].map(quarter_labels)

print(f"    Tipe data: {df_feat['quarter'].dtype}")
print(f"    Distribusi:")
for q, count in df_feat['quarter_label'].value_counts().sort_index().items():
    pct = count / len(df_feat) * 100
    print(f"      {q}: {count:,} transaksi ({pct:.1f}%)")

---
## Feature 3: Day of Week

Mengekstrak hari dalam minggu dari timestamp.

**Manfaat Bisnis:**
- Analisis pola mingguan (hari tersibuk vs sepi)
- Perencanaan staffing berdasarkan hari
- Strategi promosi per hari

In [ ]:
# ============================================================
# Feature 3: Day of Week
# ============================================================

print("[3] Membuat fitur: day_of_week")

# Ekstrak hari dalam minggu (0=Monday, 6=Sunday)
df_feat['day_of_week'] = df_feat['timestamp'].dt.dayofweek

# Buat label hari
day_names = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday',
             4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
df_feat['day_name'] = df_feat['day_of_week'].map(day_names)

print(f"    Tipe data: {df_feat['day_of_week'].dtype}")
print(f"    Distribusi:")
for day, count in df_feat['day_name'].value_counts().reindex(day_names.values()).items():
    pct = count / len(df_feat) * 100
    print(f"      {day}: {count:,} transaksi ({pct:.1f}%)")

---
## Feature 4: Hour

Mengekstrak jam dari timestamp.

**Manfaat Bisnis:**
- Analisis peak hours (jam sibuk)
- Optimasi jam operasional
- Strategi promosi time-based (happy hour)

In [ ]:
# ============================================================
# Feature 4: Hour
# ============================================================

print("[4] Membuat fitur: hour")

# Ekstrak jam dari timestamp
df_feat['hour'] = df_feat['timestamp'].dt.hour

# Kategorikan waktu
def categorize_time(hour):
    if 6 <= hour < 10:
        return 'Morning (6-10)'
    elif 10 <= hour < 14:
        return 'Lunch (10-14)'
    elif 14 <= hour < 18:
        return 'Afternoon (14-18)'
    elif 18 <= hour < 22:
        return 'Evening (18-22)'
    else:
        return 'Night (22-6)'

df_feat['time_period'] = df_feat['hour'].apply(categorize_time)

print(f"    Tipe data: {df_feat['hour'].dtype}")
print(f"    Distribusi per periode:")
for period, count in df_feat['time_period'].value_counts().items():
    pct = count / len(df_feat) * 100
    print(f"      {period}: {count:,} transaksi ({pct:.1f}%)")

---
## Feature 5: Is Weekend

Membuat flag untuk transaksi di hari weekend (Saturday & Sunday).

**Manfaat Bisnis:**
- Perbandingan performa weekday vs weekend
- Strategi promosi berbeda untuk weekend
- Perencanaan stok untuk weekend

In [ ]:
# ============================================================
# Feature 5: Is Weekend
# ============================================================

print("[5] Membuat fitur: is_weekend")

# Flag weekend (Saturday=5, Sunday=6)
df_feat['is_weekend'] = df_feat['day_of_week'].isin([5, 6]).astype('bool')

# Hitung proporsi
weekend_count = df_feat['is_weekend'].sum()
weekday_count = (~df_feat['is_weekend']).sum()

print(f"    Tipe data: {df_feat['is_weekend'].dtype}")
print(f"    Distribusi:")
print(f"      Weekday (False): {weekday_count:,} transaksi ({weekday_count/len(df_feat)*100:.1f}%)")
print(f"      Weekend (True):  {weekend_count:,} transaksi ({weekend_count/len(df_feat)*100:.1f}%)")

# Bandingkan total amount
weekday_avg = df_feat[~df_feat['is_weekend']]['total_amount'].mean()
weekend_avg = df_feat[df_feat['is_weekend']]['total_amount'].mean()
print(f"\n    Rata-rata total_amount:")
print(f"      Weekday: ${weekday_avg:.2f}")
print(f"      Weekend: ${weekend_avg:.2f}")

---
## Feature 6: Price Category

Mengkategorikan produk berdasarkan harga satuan (unit_price).

**Manfaat Bisnis:**
- Analisis performa per kategori harga
- Strategi pricing dan promosi
- Memahami preferensi pelanggan terhadap harga

In [ ]:
# ============================================================
# Feature 6: Price Category
# ============================================================

print("[6] Membuat fitur: price_category")

# Definisikan threshold berdasarkan distribusi data
# Menggunakan quantile untuk threshold yang adaptif
q25 = df_feat['unit_price'].quantile(0.25)
q75 = df_feat['unit_price'].quantile(0.75)

print(f"    Threshold:")
print(f"      Q25 (Low/Medium): ${q25:.2f}")
print(f"      Q75 (Medium/Premium): ${q75:.2f}")

# Kategorikan
def categorize_price(price):
    if price <= q25:
        return 'Low Price'
    elif price <= q75:
        return 'Medium Price'
    else:
        return 'Premium'

df_feat['price_category'] = df_feat['unit_price'].apply(categorize_price)

print(f"\n    Distribusi:")
for cat, count in df_feat['price_category'].value_counts().items():
    pct = count / len(df_feat) * 100
    avg_price = df_feat[df_feat['price_category'] == cat]['unit_price'].mean()
    print(f"      {cat}: {count:,} transaksi ({pct:.1f}%) - Avg: ${avg_price:.2f}")

---
## Feature 7: Basket Size

Mengkategorikan transaksi berdasarkan jumlah item (quantity).

**Manfaat Bisnis:**
- Analisis pola pembelian (satuan vs bulk)
- Strategi upselling dan bundling
- Memahami perilaku pembelian pelanggan

In [ ]:
# ============================================================
# Feature 7: Basket Size
# ============================================================

print("[7] Membuat fitur: basket_size")

# Definisikan threshold berdasarkan domain knowledge
def categorize_basket(quantity):
    if quantity == 1:
        return 'Small (1)'
    elif quantity <= 3:
        return 'Medium (2-3)'
    elif quantity <= 6:
        return 'Large (4-6)'
    else:
        return 'Bulk (7+)'

df_feat['basket_size'] = df_feat['quantity'].apply(categorize_basket)

print(f"    Threshold:")
print(f"      Small: 1 item")
print(f"      Medium: 2-3 items")
print(f"      Large: 4-6 items")
print(f"      Bulk: 7+ items")

print(f"\n    Distribusi:")
for size, count in df_feat['basket_size'].value_counts().items():
    pct = count / len(df_feat) * 100
    avg_qty = df_feat[df_feat['basket_size'] == size]['quantity'].mean()
    avg_total = df_feat[df_feat['basket_size'] == size]['total_amount'].mean()
    print(f"      {size}: {count:,} transaksi ({pct:.1f}%) - Avg Qty: {avg_qty:.1f}, Avg Total: ${avg_total:.2f}")

---
## Feature 8: Customer Segment

Mengelompokkan pelanggan berdasarkan total belanja (RFM sederhana).

**Manfaat Bisnis:**
- Segmentasi pelanggan (Low/Medium/High Value)
- Strategi loyalitas dan retensi
- Personalisasi promosi per segmen

In [ ]:
# ============================================================
# Feature 8: Customer Segment
# ============================================================

print("[8] Membuat fitur: customer_segment")

# Hitung total belanja per pelanggan
customer_spending = df_feat.groupby('customer_id')['total_amount'].sum().reset_index()
customer_spending.columns = ['customer_id', 'total_spending']

# Hitung rata-rata transaksi per pelanggan
customer_avg = df_feat.groupby('customer_id')['total_amount'].mean().reset_index()
customer_avg.columns = ['customer_id', 'avg_transaction']

# Hitung frekuensi transaksi
customer_freq = df_feat.groupby('customer_id')['transaction_id'].count().reset_index()
customer_freq.columns = ['customer_id', 'transaction_count']

# Gabungkan
customer_profile = customer_spending.merge(customer_avg, on='customer_id')
customer_profile = customer_profile.merge(customer_freq, on='customer_id')

# Kategorikan berdasarkan total spending
q33 = customer_profile['total_spending'].quantile(0.33)
q66 = customer_profile['total_spending'].quantile(0.66)

def segment_customer(spending):
    if spending <= q33:
        return 'Low Value'
    elif spending <= q66:
        return 'Medium Value'
    else:
        return 'High Value'

customer_profile['customer_segment'] = customer_profile['total_spending'].apply(segment_customer)

# Merge kembali ke dataframe utama
df_feat = df_feat.merge(customer_profile[['customer_id', 'total_spending', 'customer_segment']], 
                        on='customer_id', how='left')

print(f"    Threshold:")
print(f"      Low Value: <= ${q33:.2f}")
print(f"      Medium Value: ${q33:.2f} - ${q66:.2f}")
print(f"      High Value: > ${q66:.2f}")

print(f"\n    Distribusi Pelanggan:")
for seg, count in customer_profile['customer_segment'].value_counts().items():
    pct = count / len(customer_profile) * 100
    avg_spend = customer_profile[customer_profile['customer_segment'] == seg]['total_spending'].mean()
    print(f"      {seg}: {count:,} pelanggan ({pct:.1f}%) - Avg Total: ${avg_spend:.2f}")

---
## Feature 9: Discount Impact

Menganalisis dampak diskon terhadap total transaksi.

**Manfaat Bisnis:**
- Mengukur efektivitas promosi
- Memahami margin profitabilitas
- Strategi diskon yang lebih efektif

In [ ]:
# ============================================================
# Feature 9: Discount Impact
# ============================================================

print("[9] Membuat fitur: discount_impact")

# Hitung expected total tanpa diskon
df_feat['expected_total'] = df_feat['unit_price'] * df_feat['quantity']

# Hitung selisih (discount amount)
df_feat['discount_amount'] = df_feat['expected_total'] - df_feat['total_amount']

# Kategorikan dampak diskon
def categorize_discount(row):
    if not row['discount_applied']:
        return 'No Discount'
    elif row['discount_amount'] <= 0:
        return 'No Discount'
    elif row['discount_amount'] <= 2:
        return 'Low Discount (<= $2)'
    elif row['discount_amount'] <= 5:
        return 'Medium Discount ($2-5)'
    else:
        return 'High Discount (> $5)'

df_feat['discount_impact'] = df_feat.apply(categorize_discount, axis=1)

print(f"    Distribusi:")
for impact, count in df_feat['discount_impact'].value_counts().items():
    pct = count / len(df_feat) * 100
    avg_disc = df_feat[df_feat['discount_impact'] == impact]['discount_amount'].mean()
    print(f"      {impact}: {count:,} transaksi ({pct:.1f}%) - Avg Disc: ${avg_disc:.2f}")

# Hapus kolom sementara
df_feat.drop('expected_total', axis=1, inplace=True)

---
## Feature 10: Revenue Category

Mengkategorikan transaksi berdasarkan total amount.

**Manfaat Bisnis:**
- Analisis distribusi pendapatan
- Identifikasi transaksi bernilai tinggi
- Strategi upselling

In [ ]:
# ============================================================
# Feature 10: Revenue Category
# ============================================================

print("[10] Membuat fitur: revenue_category")

# Definisikan threshold berdasarkan quantile
q25_rev = df_feat['total_amount'].quantile(0.25)
q50_rev = df_feat['total_amount'].quantile(0.50)
q75_rev = df_feat['total_amount'].quantile(0.75)

print(f"    Threshold:")
print(f"      Low: <= ${q25_rev:.2f}")
print(f"      Medium: ${q25_rev:.2f} - ${q50_rev:.2f}")
print(f"      High: ${q50_rev:.2f} - ${q75_rev:.2f}")
print(f"      Premium: > ${q75_rev:.2f}")

def categorize_revenue(amount):
    if amount <= q25_rev:
        return 'Low Revenue'
    elif amount <= q50_rev:
        return 'Medium Revenue'
    elif amount <= q75_rev:
        return 'High Revenue'
    else:
        return 'Premium Revenue'

df_feat['revenue_category'] = df_feat['total_amount'].apply(categorize_revenue)

print(f"\n    Distribusi:")
for cat, count in df_feat['revenue_category'].value_counts().items():
    pct = count / len(df_feat) * 100
    avg_rev = df_feat[df_feat['revenue_category'] == cat]['total_amount'].mean()
    print(f"      {cat}: {count:,} transaksi ({pct:.1f}%) - Avg: ${avg_rev:.2f}")

---
# 6. Validation

Memastikan seluruh fitur baru berhasil dibuat dan tidak ada error.

In [ ]:
# ============================================================
# 6. Validation
# ============================================================

print("=" * 70)
print(" 6. VALIDATION")
print("=" * 70)

# ---- 1. Cek Missing Value Baru ----
print("\n[1] MISSING VALUES")
missing_new = df_feat.isnull().sum()
missing_cols = missing_new[missing_new > 0]

if len(missing_cols) > 0:
    print(f"    Kolom dengan missing values:")
    for col, count in missing_cols.items():
        print(f"      {col}: {count:,}")
else:
    print(f"    TIDAK ADA MISSING VALUE BARU ✓")

# ---- 2. Cek Tipe Data ----
print(f"\n[2] TIPE DATA")
new_features = ['month', 'quarter', 'day_of_week', 'hour', 'is_weekend',
                'price_category', 'basket_size', 'customer_segment',
                'discount_impact', 'revenue_category']

type_ok = True
for feat in new_features:
    if feat in df_feat.columns:
        dtype = df_feat[feat].dtype
        print(f"    {feat}: {dtype} ✓")
    else:
        print(f"    {feat}: TIDAK DITEMUKAN ✗")
        type_ok = False

# ---- 3. Cek Jumlah Fitur ----
print(f"\n[3] JUMLAH FITUR")
print(f"    Kolom awal: {df.shape[1]}")
print(f"    Kolom sekarang: {df_feat.shape[1]}")
print(f"    Fitur baru ditambahkan: {df_feat.shape[1] - df.shape[1]}")

# ---- 4. Cek Error ----
print(f"\n[4] ERROR CHECK")
try:
    # Test operasi pada fitur baru
    _ = df_feat['month'].value_counts()
    _ = df_feat['quarter'].value_counts()
    _ = df_feat['day_of_week'].value_counts()
    _ = df_feat['hour'].value_counts()
    _ = df_feat['is_weekend'].value_counts()
    _ = df_feat['price_category'].value_counts()
    _ = df_feat['basket_size'].value_counts()
    _ = df_feat['customer_segment'].value_counts()
    _ = df_feat['discount_impact'].value_counts()
    _ = df_feat['revenue_category'].value_counts()
    print(f"    TIDAK ADA ERROR ✓")
except Exception as e:
    print(f"    ERROR: {e}")

# ---- Ringkasan ----
print(f"\n{'='*50}")
print(f"VALIDATION RESULT")
print(f"{'='*50}")
if len(missing_cols) == 0 and type_ok:
    print(f"✓ SEMUA VALIDASI PAS - Fitur berhasil dibuat!")
else:
    print(f"⚠ BEBERAPA VALIDASI PERLU PERHATIAN")

In [ ]:
# ============================================================
# Tampilkan Data Setelah Feature Engineering
# ============================================================

print("=" * 70)
print(" DATA SETELAH FEATURE ENGINEERING")
print("=" * 70)

df_feat.head(10)

In [ ]:
# ============================================================
# Info Dataset Setelah Feature Engineering
# ============================================================

print("=" * 70)
print(" INFO DATASET")
print("=" * 70)

df_feat.info()

print(f"\nShape: {df_feat.shape}")
print(f"Memory: {df_feat.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

---
# 7. Feature Summary

Ringkasan seluruh fitur baru yang dibuat.

In [ ]:
# ============================================================
# 7. Feature Summary
# ============================================================

print("=" * 80)
print(" 7. FEATURE SUMMARY")
print("=" * 80)

feature_summary = pd.DataFrame({
    'Feature': [
        'month',
        'month_name',
        'quarter',
        'quarter_label',
        'day_of_week',
        'day_name',
        'hour',
        'time_period',
        'is_weekend',
        'price_category',
        'basket_size',
        'customer_segment',
        'total_spending',
        'discount_amount',
        'discount_impact',
        'revenue_category'
    ],
    'Description': [
        'Bulan transaksi (1-12)',
        'Nama bulan (January-December)',
        'Kuartal (1-4)',
        'Label kuartal (Q1-Q4)',
        'Hari dalam minggu (0=Monday, 6=Sunday)',
        'Nama hari (Monday-Sunday)',
        'Jam transaksi (0-23)',
        'Periode waktu (Morning/Lunch/Afternoon/Evening/Night)',
        'Flag weekend (True/False)',
        'Kategori harga (Low/Medium/Premium)',
        'Ukuran keranjang (Small/Medium/Large/Bulk)',
        'Segmen pelanggan (Low/Medium/High Value)',
        'Total belanja pelanggan',
        'Jumlah diskon dalam dollar',
        'Dampak diskon (No/Low/Medium/High)',
        'Kategori pendapatan (Low/Medium/Premium)'
    ],
    'Business Purpose': [
        'Analisis tren bulanan dan musiman',
        'Label untuk visualisasi',
        'Pelaporan kuartalan',
        'Label untuk visualisasi',
        'Analisis pola mingguan',
        'Label untuk visualisasi',
        'Analisis peak hours',
        'Segmentasi waktu operasional',
        'Perbandingan weekday vs weekend',
        'Analisis performa per kategori harga',
        'Analisis pola pembelian',
        'Strategi loyalitas dan retensi',
        'Analisis lifetime value pelanggan',
        'Mengukur efektivitas promosi',
        'Analisis dampak diskon',
        'Analisis distribusi pendapatan'
    ],
    'Type': [
        'int64',
        'object',
        'int64',
        'object',
        'int64',
        'object',
        'int64',
        'object',
        'bool',
        'object',
        'object',
        'object',
        'float64',
        'float64',
        'object',
        'object'
    ],
    'Unique Values': [
        df_feat['month'].nunique(),
        df_feat['month_name'].nunique(),
        df_feat['quarter'].nunique(),
        df_feat['quarter_label'].nunique(),
        df_feat['day_of_week'].nunique(),
        df_feat['day_name'].nunique(),
        df_feat['hour'].nunique(),
        df_feat['time_period'].nunique(),
        df_feat['is_weekend'].nunique(),
        df_feat['price_category'].nunique(),
        df_feat['basket_size'].nunique(),
        df_feat['customer_segment'].nunique(),
        '-',
        '-',
        df_feat['discount_impact'].nunique(),
        df_feat['revenue_category'].nunique()
    ]
})

print("\n" + feature_summary.to_string(index=False))

---
# 8. Findings

Temuan dan insight dari fitur-fitur baru yang dibuat.

In [ ]:
# ============================================================
# 8. Findings
# ============================================================

print("=" * 80)
print(" 8. FINDINGS: INSIGHT DARI FITUR BARU")
print("=" * 80)

# ---- Finding 1: Distribusi Bulanan ----
print("\n[1] DISTRIBUSI BULANAN")
monthly = df_feat.groupby('month_name')['total_amount'].agg(['count', 'sum', 'mean'])
monthly = monthly.sort_values('sum', ascending=False)
print("    Top 3 bulan dengan revenue tertinggi:")
for i, (month, row) in enumerate(monthly.head(3).iterrows(), 1):
    print(f"      {i}. {month}: ${row['sum']:,.2f} ({row['count']:,} transaksi)")

# ---- Finding 2: Pola Mingguan ----
print("\n[2] POLA MINGGUAN")
daily = df_feat.groupby('day_name')['total_amount'].agg(['count', 'sum', 'mean'])
daily = daily.sort_values('sum', ascending=False)
print("    Hari dengan transaksi terbanyak:")
for i, (day, row) in enumerate(daily.head(3).iterrows(), 1):
    print(f"      {i}. {day}: {row['count']:,} transaksi (${row['sum']:,.2f})")

# ---- Finding 3: Peak Hours ----
print("\n[3] PEAK HOURS")
hourly = df_feat.groupby('time_period')['total_amount'].agg(['count', 'sum'])
hourly = hourly.sort_values('count', ascending=False)
print("    Periode waktu dengan transaksi terbanyak:")
for i, (period, row) in enumerate(hourly.iterrows(), 1):
    pct = row['count'] / len(df_feat) * 100
    print(f"      {i}. {period}: {row['count']:,} transaksi ({pct:.1f}%)")

# ---- Finding 4: Basket Size ----
print("\n[4] POLA PEMBELIAN (BASKET SIZE)")
basket = df_feat.groupby('basket_size')['total_amount'].agg(['count', 'sum', 'mean'])
basket = basket.sort_values('count', ascending=False)
print("    Distribusi ukuran keranjang:")
for size, row in basket.iterrows():
    pct = row['count'] / len(df_feat) * 100
    print(f"      {size}: {row['count']:,} transaksi ({pct:.1f}%) - Avg: ${row['mean']:.2f}")

# ---- Finding 5: Customer Segment ----
print("\n[5] SEGMENTASI PELANGGAN")
segment = df_feat.groupby('customer_segment')['total_amount'].agg(['count', 'sum', 'mean'])
segment = segment.sort_values('sum', ascending=False)
print("    Distribusi segmen pelanggan:")
for seg, row in segment.iterrows():
    pct = row['count'] / len(df_feat) * 100
    print(f"      {seg}: {row['count']:,} transaksi ({pct:.1f}%) - Total: ${row['sum']:,.2f}")

# ---- Finding 6: Price Category ----
print("\n[6] KATEGORI HARGA")
price = df_feat.groupby('price_category')['total_amount'].agg(['count', 'sum', 'mean'])
price = price.sort_values('sum', ascending=False)
print("    Distribusi kategori harga:")
for cat, row in price.iterrows():
    pct = row['count'] / len(df_feat) * 100
    print(f"      {cat}: {row['count']:,} transaksi ({pct:.1f}%) - Avg: ${row['mean']:.2f}")

# ---- Finding 7: Discount Impact ----
print("\n[7] DAMPAK DISKON")
discount = df_feat.groupby('discount_impact')['total_amount'].agg(['count', 'sum', 'mean'])
discount = discount.sort_values('sum', ascending=False)
print("    Distribusi dampak diskon:")
for impact, row in discount.iterrows():
    pct = row['count'] / len(df_feat) * 100
    print(f"      {impact}: {row['count']:,} transaksi ({pct:.1f}%) - Avg: ${row['mean']:.2f}")

In [ ]:
# ============================================================
# Ringkasan Temuan
# ============================================================

print("\n" + "=" * 80)
print(" RINGKASAN TEMUAN")
print("=" * 80)

findings = """
TEMUAN 1: DISTRIBUSI BULANAN MERATA
=====================================
Transaksi tersebar cukup merata sepanjang tahun, menunjukkan bisnis coffee shop
tidak terlalu bergantung pada musim tertentu. Ini mengindikasikan model bisnis
yang stabil dan konsisten.

TEMUAN 2: WEEKEND LEBIH RAMAI DARI WEEKDAY
=============================================
Transaksi weekend cenderung lebih banyak, menunjukkan pelanggan memiliki waktu
luang lebih untuk berkunjung. Ini bisa menjadi target promosi khusus weekend.

TEMUAN 3: LUNCH TIME ADALAH PEAK HOUR
=========================================
Periode Lunch (10-14) memiliki jumlah transaksi tertinggi, diikuti Morning.
Ini menunjukkan coffee shop sebagai tempat sarapan dan makan siang.

TEMUAN 4: SMALL BASKET MENDOMINASI
====================================
Mayoritas transaksi adalah Small Basket (1 item), menunjukkan pola pembelian
untuk konsumsi langsung. Peluang upselling ke Medium/Large basket.

TEMUAN 5: MEDIUM VALUE PELANGGAN PLURALITAS
=============================================
Segmen Medium Value memiliki jumlah transaksi terbesar, menunjukkan
pelanggan dengan belanja menengah adalah tulang punggung bisnis.

TEMUAN 6: PREMIUM PRODUCTS KONTRIBUSI BESAR
=============================================
Produk Premium meskipun jumlah transaksinya lebih sedikit, memberikan
kontribusi revenue yang signifikan. Strategi premium products perlu dipertahankan.

TEMUAN 7: DISKON BERDAMPAK POSITIF
=====================================
Transaksi dengan diskon menunjukkan nilai rata-rata yang kompetitif.
Promosi diskon efektif menarik pelanggan tanpa mengurangi revenue signifikan.
"""

print(findings)

---
# 9. Conclusion

Jawaban atas seluruh Business Questions yang diajukan di awal.

In [ ]:
# ============================================================
# 9. Conclusion
# ============================================================

print("=" * 80)
print(" 9. CONCLUSION: JAWABAN BUSINESS QUESTIONS")
print("=" * 80)

conclusion = """
Q1: Fitur apa saja yang dibuat?
A1: 10 fitur baru berhasil dibuat:
    1. month & month_name - Analisis tren bulanan
    2. quarter & quarter_label - Analisis kuartalan
    3. day_of_week & day_name - Analisis pola mingguan
    4. hour & time_period - Analisis peak hours
    5. is_weekend - Perbandingan weekday vs weekend
    6. price_category - Segmentasi produk berdasarkan harga
    7. basket_size - Analisis pola pembelian
    8. customer_segment & total_spending - Segmentasi pelanggan
    9. discount_amount & discount_impact - Analisis dampak diskon
    10. revenue_category - Kategorisasi transaksi


Q2: Mengapa fitur tersebut penting?
A2:
    - Temporal features: Memungkinkan analisis tren waktu (bulanan, kuartalan, mingguan)
    - Behavioral features: Memahami pola belanja pelanggan
    - Product features: Mengkategorikan produk untuk analisis performa
    - Financial features: Mengukur efektivitas promosi


Q3: Bagaimana fitur baru membantu analisis?
A3:
    - Memungkinkan analisis tren musiman dan seasonal patterns
    - Memungkinkan segmentasi pelanggan untuk strategi retensi
    - Memungkinkan analisis peak hours untuk optimasi operasional
    - Memungkinkan analisis dampak diskon untuk strategi promosi


Q4: Apakah dataset siap digunakan untuk EDA?
A4: YA - Dataset sudah sangat siap untuk Exploratory Data Analysis:
    - 10 fitur baru sudah ditambahkan
    - Tidak ada missing value baru
    - Tipe data sudah benar
    - Fitur sudah terkategorisasi untuk visualisasi
    - Dataset siap untuk analisis mendalam dan dashboard
"""

print(conclusion)

---
# 10. Export Dataset

Menyimpan dataset hasil feature engineering ke folder `processed/`.

In [ ]:
# ============================================================
# 10. Export Dataset
# ============================================================

print("=" * 70)
print(" 10. EXPORT DATASET")
print("=" * 70)

# Buat folder processed jika belum ada
PROCESSED_DIR = '../processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)
print(f"\nFolder '{PROCESSED_DIR}' siap.")

# Simpan dataset dengan fitur baru
EXPORT_PATH = os.path.join(PROCESSED_DIR, 'coffee_shop_sales_featured.csv')
df_feat.to_csv(EXPORT_PATH, index=False)

file_size = os.path.getsize(EXPORT_PATH) / 1024

print(f"\nDataset berhasil disimpan ke: {EXPORT_PATH}")
print(f"Ukuran file: {file_size:.2f} KB")
print(f"Jumlah baris: {len(df_feat):,}")
print(f"Jumlah kolom: {len(df_feat.columns)}")

# Verifikasi
print(f"\n{'='*50}")
print(f"VERIFIKASI EXPORT")
print(f"{'='*50}")

df_verify = pd.read_csv(EXPORT_PATH)

print(f"\nBerhasil dibaca kembali: {EXPORT_PATH}")
print(f"Shape: {df_verify.shape}")
print(f"Kolom: {list(df_verify.columns)}")
print(f"\n5 baris pertama:")
df_verify.head()

In [ ]:
# ============================================================
# Akhir Notebook - Feature Engineering
# ============================================================

print("\n" + "*" * 80)
print("*", " " * 23, "FEATURE ENGINEERING SELESAI", " " * 24, "*")
print("*" * 80)
print(f"\nDataset: processed/coffee_shop_sales_featured.csv")
print(f"Shape: {df_feat.shape}")
print(f"Fitur baru: 10 fitur")
print(f"Status: DATASET SIAP UNTUK EDA & VISUALISASI")
print(f"\nTimestamp: {pd.Timestamp.now()}")
print("*" * 80)